# 01 — Modelo Baseline para Classificação

Este notebook apresenta um template didático para criação de um modelo baseline de **classificação**.

Use este notebook quando o objetivo for prever uma **classe**, por exemplo:

- cliente compra ou não compra;
- transação é fraude ou não fraude;
- aluno aprovado ou reprovado;
- risco baixo, médio ou alto;
- espécie de flor.

## Premissa

Este template pressupõe que o dataset já passou pela etapa de preparação dos dados.

Ou seja:

- não há valores nulos relevantes;
- as variáveis categóricas já foram tratadas;
- as variáveis preditoras estão prontas para o modelo;
- a variável-alvo já está definida.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 1. Carregar o dataset

Ajuste o caminho do arquivo conforme o local onde o dataset estiver salvo.

Exemplos de datasets didáticos para classificação:

- Titanic Dataset — alvo: `Survived`
- Iris Species — alvo: `Species`
- Pima Indians Diabetes — alvo: `Outcome`

In [ ]:
df = pd.read_csv("dataset_classificacao.csv")

df.head()

## 2. Visualização inicial dos dados

Mesmo com os dados já preparados, é importante entender a estrutura do dataset.

In [ ]:
print("Linhas e colunas:", df.shape)

df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

## 3. Visualização gráfica

Os gráficos ajudam a entender distribuição, dispersão e possíveis outliers.

In [ ]:
df.select_dtypes(include=np.number).hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
df.select_dtypes(include=np.number).plot(
    kind="box",
    figsize=(12, 6),
    rot=45
)

plt.title("Boxplot das variáveis numéricas")
plt.show()

In [ ]:
correlacao = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(10, 6))
plt.imshow(correlacao, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=90)
plt.yticks(range(len(correlacao.columns)), correlacao.columns)
plt.title("Matriz de correlação")
plt.show()

## 4. Separar variáveis preditoras e variável-alvo

A variável-alvo é a coluna que o modelo deve aprender a prever.

In [ ]:
target = "alvo"  # Altere para o nome da coluna alvo

X = df.drop(columns=[target])
y = df[target]

print("Variáveis preditoras:")
print(X.head())

print("\nVariável-alvo:")
print(y.head())

## 5. Analisar distribuição das classes

Esta etapa é importante para identificar desbalanceamento.

Exemplo: se 95% dos registros pertencem a uma classe, a acurácia pode ser enganosa.

In [ ]:
print("Contagem por classe:")
print(y.value_counts())

print("\nProporção por classe:")
print(y.value_counts(normalize=True))

## 6. Dividir treino e teste

Usamos `stratify=y` para manter a proporção das classes no treino e no teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 7. Normalização

A normalização é importante para modelos sensíveis à escala, como:

- Regressão Logística;
- KNN;
- SVM;
- Redes Neurais.

Para árvores de decisão, normalização normalmente não é obrigatória.

Regra importante:

- `fit_transform` apenas no treino;
- `transform` no teste.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Outras opções de normalização

Use uma das opções abaixo conforme o objetivo didático.

In [ ]:
# MinMaxScaler: coloca os valores normalmente entre 0 e 1
# scaler = MinMaxScaler()

# RobustScaler: mais robusto a outliers
# scaler = RobustScaler()

## 8. Baseline ingênuo

O `DummyClassifier` cria um modelo extremamente simples.

Ele serve como referência mínima para comparar o modelo real.

A estratégia `most_frequent` sempre prevê a classe mais frequente.

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

print("Acurácia do baseline ingênuo:", accuracy_score(y_test, y_pred_dummy))

## 9. Modelo baseline real

Para fins didáticos, a árvore de decisão é uma boa escolha inicial.

Hiperparâmetros usados:

- `max_depth`: profundidade máxima da árvore;
- `min_samples_split`: mínimo de amostras para dividir um nó;
- `random_state`: garante reprodutibilidade.

In [ ]:
model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=10,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## 10. Avaliação do modelo de classificação

Métricas principais:

- **Acurácia**: percentual geral de acertos.
- **Precisão**: das previsões positivas, quantas estavam corretas.
- **Recall**: dos positivos reais, quantos foram encontrados.
- **F1-score**: equilíbrio entre precisão e recall.
- **Matriz de confusão**: mostra onde o modelo acertou e errou.

In [ ]:
print("=== Baseline Ingênuo ===")
print("Acurácia:", accuracy_score(y_test, y_pred_dummy))

print("\n=== Modelo Baseline Real ===")
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred, zero_division=0))

## 11. Matriz de confusão

A matriz de confusão compara os valores reais com os valores previstos.

Regra didática:

- a diagonal principal representa os acertos;
- os valores fora da diagonal representam os erros.

Em classificação binária:

- **TP**: era positivo e o modelo previu positivo;
- **TN**: era negativo e o modelo previu negativo;
- **FP**: era negativo, mas o modelo previu positivo;
- **FN**: era positivo, mas o modelo previu negativo.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=model.classes_
)

disp.plot()
plt.title("Matriz de Confusão")
plt.show()

## 12. Percentual de acerto

Em classificação, o percentual de acerto corresponde à acurácia.

In [ ]:
percentual_acerto = accuracy_score(y_test, y_pred) * 100

print(f"Percentual de acerto no teste: {percentual_acerto:.2f}%")

## 13. Predição com uma amostra real do teste

Esta etapa ajuda o aluno a entender como o modelo usa um registro para gerar uma previsão.

In [ ]:
vetor_teste = X_test.iloc[[0]]

predicao = model.predict(vetor_teste)
valor_real = y_test.iloc[0]

print("Vetor de teste:")
print(vetor_teste)

print("\nPredição:", predicao[0])
print("Valor real:", valor_real)
print("Acertou?", predicao[0] == valor_real)

## 14. Predição com vetor manual

Aqui criamos um registro simples usando mediana para variáveis numéricas e moda para variáveis categóricas.

Como a premissa é que o dataset já está preparado, espera-se que `X` já esteja pronto para o modelo.

In [ ]:
novo_registro = pd.DataFrame([{
    col: X[col].median() if np.issubdtype(X[col].dtype, np.number) else X[col].mode()[0]
    for col in X.columns
}])

predicao_novo = model.predict(novo_registro)

print("Novo registro:")
print(novo_registro)

print("\nClasse prevista:", predicao_novo[0])

if hasattr(model, "predict_proba"):
    probabilidades = model.predict_proba(novo_registro)

    df_probabilidades = pd.DataFrame(
        probabilidades,
        columns=model.classes_
    )

    print("\nProbabilidades por classe:")
    print(df_probabilidades)

## Sugestões didáticas

1. Sempre compare o modelo real com o `DummyClassifier`.
2. Não avalie classificação apenas com acurácia.
3. Use matriz de confusão para mostrar os tipos de erro.
4. Explique precisão e recall usando exemplos de negócio.
5. Mostre que classes desbalanceadas podem enganar a avaliação.